**Module:** 02 Mathematics  
**Part:** E — Loss Functions  
**Duration:** ~2.5 hours

---

## Learning Objectives

1. Derive Dice loss
2. Derive IoU (Jaccard) loss
3. Understand Lovász loss extension
4. Connect to water-bodies-detection AquaBoundaryLoss


## 1. Dice Coefficient and Loss

$$\text{Dice} = \frac{2|A \cap B|}{|A| + |B|} = \frac{2\sum p_i g_i}{\sum p_i + \sum g_i}$$

$$L_{\text{Dice}} = 1 - \text{Dice}$$

Good for **class imbalance** (small water bodies in large tiles).

## 2. IoU (Jaccard) Loss

$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|} = \frac{\sum p_i g_i}{\sum p_i + \sum g_i - \sum p_i g_i}$$

$$L_{\text{IoU}} = 1 - \text{IoU}$$

## 3. Your water-bodies-detection Loss

From `losses.py`:
```python
AquaBoundaryLoss = w_aqua * (BCE_aqua + Dice_aqua) + w_boundary * (BCE_boundary + Dice_boundary)
```

Each head gets BCE (pixel-wise classification) + Dice (region overlap). This handles both pixel accuracy and region-level segmentation quality.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11
rng = np.random.default_rng(42)

In [ ]:
def dice_coefficient(pred, target, eps=1e-7):
    pred = pred.flatten().astype(float)
    target = target.flatten().astype(float)
    intersection = (pred * target).sum()
    return (2 * intersection + eps) / (pred.sum() + target.sum() + eps)

def iou_coefficient(pred, target, eps=1e-7):
    pred = pred.flatten().astype(float)
    target = target.flatten().astype(float)
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + eps) / (union + eps)

# Simulated prediction vs ground truth
pred = (rng.random((64, 64)) > 0.5).astype(float)
target = (rng.random((64, 64)) > 0.7).astype(float)

print(f'Dice: {dice_coefficient(pred, target):.4f}')
print(f'IoU:  {iou_coefficient(pred, target):.4f}')
print(f'Dice Loss: {1 - dice_coefficient(pred, target):.4f}')

In [ ]:
# Visualize: imbalanced segmentation
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Small object in large image
target = np.zeros((128, 128))
target[50:70, 50:70] = 1  # small square

# Good vs bad prediction
pred_good = target.copy()
pred_good[55:65, 55:65] = 0.8  # slight error
pred_bad = np.zeros((128, 128))
pred_bad[40:80, 40:80] = 1  # too large

for ax, p, title in zip(axes, [target, pred_good, pred_bad], ['Ground Truth', 'Good Pred', 'Bad Pred']):
    ax.imshow(p, cmap='Blues'); ax.set_title(title); ax.axis('off')

print(f'Good pred — Dice: {dice_coefficient(pred_good, target):.3f}, IoU: {iou_coefficient(pred_good, target):.3f}')
print(f'Bad pred  — Dice: {dice_coefficient(pred_bad, target):.3f}, IoU: {iou_coefficient(pred_bad, target):.3f}')
plt.tight_layout(); plt.show()

## Module 02 Complete

You now have the mathematical foundation for all ML algorithms.

**Before Module 03:** Complete exercises, assignment (logistic regression gradient), quiz ≥80%, gate questions.

See: [CHEATSHEET.md](CHEATSHEET.md) | [quiz/module_02_quiz.md](quiz/module_02_quiz.md)

---

## Interview Questions

See module quiz and exercises.

## Summary

Segmentation losses handle class imbalance — critical for GeoSpatial AI.

**Notebook 23 complete.**